# GRU

### Step-1 : Load the data

In [37]:
import pandas as pd
df = pd.read_csv("../data/datasets/cleaned/cleaned_data.csv")
texts = df["text"].astype("str").tolist()
texts = texts[:50000]
texts[:10]

["['Tetsusaiga will gain the power to break the barrier.']",
 "['What are you doing?! Give it back!']",
 "['Megumi Love']",
 "['He turned into a bow and arrow!']",
 "['You should be arrested for perjury!']",
 '["She\'s dead?"]',
 "['But if ever there were a king of the creatures of the night...']",
 "['It is quite important.']",
 "['I invited her, too.']",
 '["Rika, you see, speaks less when she\'s with someone she likes."]']

### Step-2 : Tokenize and convert to sequences

In [38]:
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

In [39]:
input_sequence = []
for line in texts:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequence.append(n_gram_sequence)
input_sequence[:10]

[[7971, 53],
 [7971, 53, 2595],
 [7971, 53, 2595, 3],
 [7971, 53, 2595, 3, 194],
 [7971, 53, 2595, 3, 194, 5],
 [7971, 53, 2595, 3, 194, 5, 482],
 [7971, 53, 2595, 3, 194, 5, 482, 3],
 [7971, 53, 2595, 3, 194, 5, 482, 3, 1272],
 [7971, 53, 2595, 3, 194, 5, 482, 3, 1272, 2],
 [68, 25]]

In [40]:
max_seq_len = max(len(seq) for seq in input_sequence)
vocab_size = len(tokenizer.word_index)+1
print(max_seq_len, vocab_size)

73 23927


### Step-3 : Add padding

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
input_sequence = pad_sequences(input_sequence, maxlen = max_seq_len, padding="pre")

### Step-4 : Define inputs and outputs

In [42]:
X = input_sequence[:,:-1]
y = input_sequence[:, -1]

### Step-5 : Create the model

In [43]:
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import GRU, Dense, Embedding, Bidirectional # type: ignore

model = Sequential([
    Embedding(input_length=max_seq_len-1, input_dim=vocab_size, output_dim=128),
    GRU(128),
    Dense(vocab_size, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"], optimizer="adam")

In [44]:
history = model.fit(X, y, epochs=5, batch_size=32)

Epoch 1/5
10365/10365 [==============================] - 597s 57ms/step - loss: 5.6380 - accuracy: 0.1554
Epoch 2/5
10365/10365 [==============================] - 594s 57ms/step - loss: 4.9465 - accuracy: 0.1998
Epoch 3/5
10365/10365 [==============================] - 596s 57ms/step - loss: 4.6511 - accuracy: 0.2179
Epoch 4/5
10365/10365 [==============================] - 596s 57ms/step - loss: 4.4230 - accuracy: 0.2312
Epoch 5/5
10365/10365 [==============================] - 596s 57ms/step - loss: 4.2247 - accuracy: 0.2447


### Step-6 : Testing

In [ ]:
import numpy as np
def generate_text(seed_text, next_words):

    # loop to generate words
    for _ in range(next_words):

        # convert seed text to token sequence
        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        # pad sequence
        token_list = pad_sequences(
            [token_list],
            maxlen=max_seq_len - 1,
            padding='pre'
        )

        # predict next word probabilities
        predicted_probs = model.predict(token_list, verbose=0)

        # get word with highest probability
        predicted_word_id = np.argmax(predicted_probs, axis=-1)[0]

        # find actual word from word id
        output_word = ""

        for word, index in tokenizer.word_index.items():

            if index == predicted_word_id:

                output_word = word
                break

        # append predicted word
        seed_text += " " + output_word

    return seed_text

In [48]:
print(generate_text("i will", 20))

i will not be able to get through the <OOV> of <OOV> <OOV> to the <OOV> of the <OOV> <OOV> and <OOV>


In [49]:
print(generate_text("the world", 20))

the world is the <OOV> of the <OOV> of the <OOV> <OOV> and <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV> <OOV>


In [50]:
print(generate_text("you are", 20))

you are the only one who doesn't know about anything about that thing in the world <OOV> <OOV> <OOV> than you can
